## 1. Load and explore the Iris dataset

In [ ]:
import pandas as pd

# Load the Iris dataset from the CSV file
df = pd.read_csv('/content/iris.csv')

# Display the first 5 rows of the dataset
display(df.head())

### Dataset Information

In [ ]:
# Display basic information about the dataset
display(df.info())

### Descriptive Statistics

In [ ]:
# Display descriptive statistics
display(df.describe())

### Check for Missing Values

In [ ]:
# Check for any missing values
display(df.isnull().sum())

### Class Distribution

In [ ]:
# Check the distribution of the target variable (species)
display(df['species'].value_counts())

### Feature Distributions (Histograms)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histograms for each feature
df.hist(edgecolor='black', linewidth=1.2, figsize=(12, 8))
plt.suptitle('Distribution of Iris Features', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### Pair Plot for Feature Relationships

In [ ]:
# Create a pair plot to visualize relationships between features colored by species
sns.pairplot(df, hue='species', diag_kind='kde')
plt.suptitle('Pair Plot of Iris Features by Species', y=1.02, fontsize=16)
plt.show()

### Correlation Heatmap

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.drop('species', axis=1).corr()

# Plot the correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Iris Features')
plt.show()

## 2. Preprocess the Data

### Separate Features and Target

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Separate features (X) and target (y)
X = df.drop('species', axis=1)
y = df['species']

display(X.head())
display(y.head())

### Encode the Target Variable

In [ ]:
# Encode the target variable 'species' from categorical to numerical
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

display(f"Original species names: {label_encoder.classes_}")
display(f"Encoded target variable (first 5): {y_encoded[:5]}")

### Split Data into Training and Testing Sets

In [ ]:
# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

display(f"X_train shape: {X_train.shape}")
display(f"X_test shape: {X_test.shape}")
display(f"y_train shape: {y_train.shape}")
display(f"y_test shape: {y_test.shape}")

### Standardize Features

In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

display("Scaled X_train (first 5 rows):\n", pd.DataFrame(X_train_scaled, columns=X.columns).head())
display("Scaled X_test (first 5 rows):\n", pd.DataFrame(X_test_scaled, columns=X.columns).head())

## 3. Train a Machine Learning Model

### Model Selection and Training (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
import time

# Initialize and train a Logistic Regression model
log_reg = LogisticRegression(max_iter=200, random_state=42)

# Record training start time
start_time = time.time()

# Train the model
log_reg.fit(X_train_scaled, y_train)

# Record training end time
end_time = time.time()

training_time = end_time - start_time

display(f"Logistic Regression model trained successfully in {training_time:.4f} seconds.")
display(f"Model parameters: {log_reg.get_params()}")

## 4. Evaluate the Model Comprehensively

### Prediction and Accuracy

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on the training and test sets
y_train_pred = log_reg.predict(X_train_scaled)
y_test_pred = log_reg.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

display(f"Training Accuracy: {train_accuracy:.4f}")
display(f"Test Accuracy: {test_accuracy:.4f}")

### Classification Report

In [ ]:
# Generate classification report for the test set
class_report = classification_report(y_test, y_test_pred, target_names=label_encoder.classes_)

display("\nClassification Report (Test Set):\n", class_report)

### Confusion Matrix

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

## 5. Summary and Conclusion

In [ ]:
display(f"\nSummary:\n")
display(f"- Model used: Logistic Regression")
display(f"- Training Accuracy: {train_accuracy:.4f}")
display(f"- Test Accuracy: {test_accuracy:.4f}")
display(f"- The model successfully achieved at least 90% accuracy on the test set, fulfilling a key success criterion.")
display(f"- Preprocessing steps included separating features/target, encoding target, splitting data, and standardizing features.")
display(f"- Comprehensive evaluation metrics (accuracy, precision, recall, F1-score) and visualizations (confusion matrix) were provided.")

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Streamlit App Title
st.title('Iris Flower Species Predictor')
st.write('Enter the measurements of the Iris flower to predict its species.')

# --- Load Data and Train Model (Self-contained for Streamlit App) ---
# This section re-does the necessary steps to make the app self-contained
@st.cache_resource
def load_and_train_model():
    # Load the Iris dataset
    df = pd.read_csv('/content/iris.csv')

    # Separate features (X) and target (y)
    X = df.drop('species', axis=1)
    y = df['species']

    # Encode the target variable
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # Split data (necessary for consistent scaling and training)
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    # X_test_scaled = scaler.transform(X_test) # Not needed for app inference

    # Train Logistic Regression model
    log_reg = LogisticRegression(max_iter=200, random_state=42)
    log_reg.fit(X_train_scaled, y_train)

    return scaler, label_encoder, log_reg

scaler, label_encoder, log_reg = load_and_train_model()

# --- User Input Features ---
st.sidebar.header('Input Features')

def user_input_features():
    sepal_length = st.sidebar.slider('Sepal Length (cm)', 4.3, 7.9, 5.4)
    sepal_width = st.sidebar.slider('Sepal Width (cm)', 2.0, 4.4, 3.4)
    petal_length = st.sidebar.slider('Petal Length (cm)', 1.0, 6.9, 1.3)
    petal_width = st.sidebar.slider('Petal Width (cm)', 0.1, 2.5, 0.2)
    data = {'sepal_length': sepal_length,
            'sepal_width': sepal_width,
            'petal_length': petal_length,
            'petal_width': petal_width}
    features = pd.DataFrame(data, index=[0])
    return features

df_input = user_input_features()

st.subheader('User Input parameters')
st.write(df_input)

# --- Prediction ---
if st.button('Predict Species'):
    # Scale the input features
    scaled_input = scaler.transform(df_input)

    # Make prediction
    prediction_encoded = log_reg.predict(scaled_input)

    # Inverse transform to get the species name
    prediction_species = label_encoder.inverse_transform(prediction_encoded)

    st.subheader('Prediction')
    st.success(f'The predicted Iris species is: **{prediction_species[0]}**')

    st.subheader('About the Model')
    st.info('This prediction is based on a Logistic Regression model trained on the Iris dataset. Features were standardized.')



In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import streamlit as st
import streamlit as st
import pandas as pd

st.set_page_config(page_title="My Colab App", page_icon="🚀")

st.title("🚀 Custom My App")

# 1. Simple User Input
name = st.text_input("Enter your name:", "M.Yousaf")

if name:
    st.write(f"### Welcome, {name}!")

# 2. Data Table (Tunnels handle these well)
st.subheader("Sample Data Display")
df = pd.DataFrame({
    'Feature': ['Speed', 'Stability', 'Ease of Use'],
    'Score': [85, 70, 95]
})
st.table(df) # st.table is more stable than st.dataframe for weak tunnels

# 3. Simple Button Logic
if st.button("Click for a Surprise"):
    st.balloons()
    st.success("It works!")
st.title("Hello Streamlit!")
st.write("This is successfully running in Colab via LocalTunnel.")

st.title("My Interactive Data App")

# Add a slider
number = st.slider("Select a range", 0, 100, 50)
st.write(f"The current number is {number}")

# Add a simple chart
chart_data = pd.DataFrame(np.random.randn(20, 3), columns=["a", "b", "c"])
st.line_chart(chart_data)

In [ ]:
!pip install pyngrok
from pyngrok import ngrok

# Replace 'YOUR_AUTHTOKEN_HERE' with the token from your ngrok dashboard
!ngrok config add-authtoken 3DV8dbdb4uAvCyVDMPcaMpzLb0O_7hRVEXNmGGZYHhJXvhvAX
# Connect the tunnel
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

# Now run the streamlit app (you might need to run this in a separate cell or with &)
!streamlit run app.py

In [ ]:
%%writefile requirements.txt
pandas
scikit-learn
streamlit
numpy
matplotlib
seaborn

In [ ]:
%%writefile README.md
# 🌸 Iris Flower Classification App

This project uses Machine Learning to predict the species of an Iris flower based on its measurements.

## 🚀 Features
- **Exploratory Data Analysis**: Visualized feature distributions and correlations.
- **Model**: Trained a Logistic Regression model with 90%+ accuracy.
- **Web App**: Built an interactive UI with Streamlit.

## 🛠️ How to Run
1. Install dependencies: `pip install -r requirements.txt`
2. Run the app: `streamlit run app.py`

In [ ]:
!mkdir data

In [ ]:
!mv iris.csv data/